# Task 4: LoRA Fine-Tuning — Granulometry Classification

- **Approach A**: Standard LoRA — 18 direct labeled examples
- **Approach B**: SEAL-inspired LoRA — ~144 augmented examples via GPT-4.1

Memory strategy: one LoRA at a time. Train A → save → free → Train B → save → free → Eval from disk.

## Setup

In [ ]:
import os, json, re, time, torch, gc, random, base64
import numpy as np
from PIL import Image
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')


## Config

In [ ]:
TRAIN_DIR = '../../datasets/granulometry/train'
TEST_DIR = '../../datasets/granulometry/test'
TRAIN_MANIFEST = '../../datasets/granulometry/train_manifest.json'
TEST_MANIFEST = '../../datasets/granulometry/test_manifest.json'
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
ORIGINAL_GSD = 8.0
MAX_DIM = 1500
SEED = 42
IMAGES_PER_CLASS = 2

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
EPOCHS = 20
LR = 2e-5
BATCH_SIZE = 1
GRAD_ACCUM = 4

AZURE_ENDPOINT = 'https://ether-openai.openai.azure.com/'
DEPLOYMENT = 'gpt-4.1'
API_VERSION = '2024-12-01-preview'
# os.environ['AZURE_OPENAI_API_KEY'] = 'your-key'


## Step 1: Select Training Images

In [ ]:
random.seed(SEED)
with open(TRAIN_MANIFEST) as f:
    train_manifest = json.load(f)
by_class = defaultdict(list)
for e in train_manifest: by_class[e['class']].append(e)
selected = []
for cls in sorted(by_class):
    random.shuffle(by_class[cls])
    selected.extend(by_class[cls][:IMAGES_PER_CLASS])
print(f'{len(selected)} training images selected')
for e in selected: print(f'  {e["class"]}: {e["image"]}')


## Step 2A: Direct Training Data (18 examples)

In [ ]:
DIRECT_PROMPT = (
    'Classify this concrete aggregate photo. GSD = 8.0 px/mm.\n'
    'Grading (DIN 1045): coarse = gaps empty, uniform; medium = gaps partially filled; '
    'fine = gaps completely filled, dense packed.\n'
    'Respond with ONLY JSON: {"max_particle_size_mm": <8|16|32>, "grading": "<coarse|medium|fine>"}'
)
direct_data = []
for entry in selected:
    img_path = os.path.join(TRAIN_DIR, entry['image'])
    gt = json.dumps({'max_particle_size_mm': entry['max_particle_size_mm'], 'grading': entry['grading']})
    direct_data.append({'messages': [
        {'role': 'user', 'content': [{'type': 'image', 'image': img_path}, {'type': 'text', 'text': DIRECT_PROMPT}]},
        {'role': 'assistant', 'content': gt},
    ]})
with open('training_data_direct.jsonl', 'w') as f:
    for r in direct_data: f.write(json.dumps(r) + '\n')
print(f'Saved {len(direct_data)} examples to training_data_direct.jsonl')

# Preview
print(f'\n--- Sample (Approach A) ---')
s = direct_data[0]
print(f'Image:    {s["messages"][0]["content"][0]["image"]}')
print(f'Prompt:   {s["messages"][0]["content"][1]["text"][:120]}...')
print(f'Response: {s["messages"][1]["content"]}')


## Step 2B: SEAL-Augmented Data (~144 examples via GPT-4.1)

In [ ]:
from openai import AzureOpenAI
gpt_client = AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=os.environ.get('AZURE_OPENAI_API_KEY',''), api_version=API_VERSION)

GRADING_DEFS = 'DIN 1045: COARSE(A)=gaps empty, uniform; MEDIUM(B)=gaps partially filled; FINE(C)=gaps completely filled, dense.'

AUG_PROMPT = '''Create 8 training examples for a vision model classifying concrete aggregate.
This image is class {cls}: max_particle_size_mm={size}, grading={grading}.
{grading_defs}
GSD=8.0 px/mm (8mm=~64px, 16mm=~128px, 32mm=~256px).
Return JSON array of {{"prompt":..., "response":...}}. Vary styles: direct JSON, chain-of-thought,
visual description, contrastive (gaps test), size-focused, grading-focused, minimal.
All must have correct: max_particle_size_mm={size}, grading="{grading}". ONLY JSON array.'''

def encode_image(path):
    with open(path, 'rb') as f: return base64.b64encode(f.read()).decode('utf-8')

print(f'GPT-4.1 ready: {AZURE_ENDPOINT}')


In [ ]:
augmented_data = []
for entry in selected:
    img_path = os.path.join(TRAIN_DIR, entry['image'])
    if not os.path.exists(img_path): continue
    prompt = AUG_PROMPT.format(cls=entry['class'], size=entry['max_particle_size_mm'],
                               grading=entry['grading'], grading_defs=GRADING_DEFS)
    print(f'  {entry["class"]}: {entry["image"]}...', end=' ', flush=True)
    try:
        resp = gpt_client.chat.completions.create(model=DEPLOYMENT, max_tokens=4096, temperature=0.7,
            messages=[{'role':'user','content':[
                {'type':'image_url','image_url':{'url':f'data:image/jpeg;base64,{encode_image(img_path)}'}},
                {'type':'text','text':prompt}]}])
        raw = resp.choices[0].message.content
        raw = re.sub(r'```json\s*','',raw); raw = re.sub(r'```\s*','',raw).strip()
        exs = json.loads(raw) if raw.startswith('[') else None
        if not exs:
            m = re.search(r'\[.*\]', raw, re.DOTALL)
            exs = json.loads(m.group()) if m else None
        if exs and isinstance(exs, list):
            for ex in exs:
                augmented_data.append({'messages':[
                    {'role':'user','content':[{'type':'image','image':img_path},{'type':'text','text':ex['prompt']}]},
                    {'role':'assistant','content':ex['response']}]})
            print(f'{len(exs)} examples')
        else: print('PARSE FAILED')
    except Exception as e: print(f'ERROR: {e}')
    time.sleep(0.5)
with open('training_data_augmented.jsonl','w') as f:
    for r in augmented_data: f.write(json.dumps(r)+'\n')
print(f'\nSaved {len(augmented_data)} examples to training_data_augmented.jsonl')


In [ ]:
if augmented_data:
    print('--- Sample 1 (Approach B) ---')
    s = augmented_data[0]
    print(f'Prompt:   {s["messages"][0]["content"][1]["text"][:200]}')
    print(f'Response: {s["messages"][1]["content"][:300]}')
    if len(augmented_data) > 2:
        print(f'\n--- Sample 2 (different style) ---')
        s2 = augmented_data[2]
        print(f'Prompt:   {s2["messages"][0]["content"][1]["text"][:200]}')
        print(f'Response: {s2["messages"][1]["content"][:300]}')


## Step 3: Load Base Model (BF16)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import get_cosine_schedule_with_warmup
from torch.utils.data import Dataset

processor = AutoProcessor.from_pretrained(MODEL_ID)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map='auto', torch_dtype=torch.bfloat16)
base_model.enable_input_require_grads()

lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, task_type='CAUSAL_LM', bias='none')

print(f'Base model loaded (BF16).')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.1f} GB')


## Step 4: Training
Trains one LoRA at a time. After training, saves adapter and frees all training memory.

In [ ]:
class GranulometryDataset(Dataset):
    def __init__(self, jsonl_path, processor):
        with open(jsonl_path) as f:
            self.data = [json.loads(line) for line in f]
        self.processor = processor
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        entry = self.data[idx]
        msgs = entry['messages']
        user_content = msgs[0]['content']
        img_path = next((c['image'] for c in user_content if c['type']=='image'), None)
        user_text = next((c['text'] for c in user_content if c['type']=='text'), '')
        assistant_text = msgs[1]['content']
        image = Image.open(img_path).convert('RGB') if img_path else None
        chat = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':user_text}]},
                {'role':'assistant','content':[{'type':'text','text':assistant_text}]}]
        text = self.processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
        inputs = self.processor(text=[text], images=[image], return_tensors='pt', padding=True)
        input_ids = inputs['input_ids'].squeeze(0)
        labels = input_ids.clone()
        # Mask user tokens
        ast_tokens = self.processor.tokenizer.encode(assistant_text, add_special_tokens=False)
        if len(ast_tokens) < len(labels): labels[:-len(ast_tokens)] = -100
        if image: image.close()
        return {'input_ids': input_ids, 'attention_mask': inputs['attention_mask'].squeeze(0),
                'labels': labels, 'pixel_values': inputs.get('pixel_values', None),
                'image_grid_thw': inputs.get('image_grid_thw', None)}


def train_lora(base_model, processor, data_path, output_dir, lora_config, epochs=EPOCHS, lr=LR):
    """Apply LoRA → train → save adapter → unload LoRA → free memory."""
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    dataset = GranulometryDataset(data_path, processor)
    print(f'Training: {len(dataset)} examples, {epochs} epochs, effective batch={GRAD_ACCUM}')
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = max(len(dataset) * epochs // GRAD_ACCUM, 1)
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps*0.1), total_steps)
    
    model.train()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0
        for i in range(len(dataset)):
            batch = dataset[i]
            ids = batch['input_ids'].unsqueeze(0).to(model.device)
            mask = batch['attention_mask'].unsqueeze(0).to(model.device)
            lab = batch['labels'].unsqueeze(0).to(model.device)
            kw = {'input_ids': ids, 'attention_mask': mask, 'labels': lab}
            if batch.get('pixel_values') is not None: kw['pixel_values'] = batch['pixel_values'].to(model.device)
            if batch.get('image_grid_thw') is not None: kw['image_grid_thw'] = batch['image_grid_thw'].to(model.device)
            out = model(**kw)
            loss = out.loss / GRAD_ACCUM
            loss.backward()
            epoch_loss += loss.item() * GRAD_ACCUM
            if (i+1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
            del ids, mask, lab, out, loss; torch.cuda.empty_cache()
        avg = epoch_loss / len(dataset)
        losses.append(avg)
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/{epochs} — loss: {avg:.4f} — lr: {scheduler.get_last_lr()[0]:.2e}')
    
    # Save adapter
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    print(f'Adapter saved to {output_dir}/')
    
    # Unload LoRA, free training memory
    model.unload()
    del model, optimizer, scheduler, dataset
    gc.collect(); torch.cuda.empty_cache()
    print(f'Memory freed. GPU 0: {torch.cuda.memory_allocated(0)/1e9:.1f} GB')
    return losses

print('Training function ready.')


### Step 4A: Train Approach A (18 direct examples)

In [ ]:
print('=== Approach A: Standard LoRA ===')
losses_a = train_lora(base_model, processor, 'training_data_direct.jsonl', 'lora_direct', lora_config)
plt.plot(losses_a); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Approach A: Training Loss'); plt.grid(True); plt.show()


### Step 4B: Train Approach B (SEAL-augmented)

In [ ]:
print('=== Approach B: SEAL-Augmented LoRA ===')
losses_b = train_lora(base_model, processor, 'training_data_augmented.jsonl', 'lora_augmented', lora_config)
plt.plot(losses_b); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Approach B: Training Loss'); plt.grid(True); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses_a, label=f'A: Direct ({len(direct_data)} ex)', color='steelblue')
ax.plot(losses_b, label=f'B: Augmented ({len(augmented_data)} ex)', color='coral')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Training Loss Comparison')
ax.legend(); ax.grid(True); plt.tight_layout(); plt.show()


## Step 5: Evaluate
Load adapters from disk one at a time to save memory.

In [ ]:
EVAL_PROMPT = '''Classify this concrete aggregate photo. GSD = {gsd:.1f} px/mm.
Grading (DIN 1045): coarse = gaps empty, uniform; medium = gaps partially filled; fine = gaps completely filled, dense.
Respond with ONLY JSON: {{"max_particle_size_mm": <8|16|32>, "grading": "<coarse|medium|fine>"}}'''

def parse_response(raw):
    if not raw: return None
    raw = re.sub(r'```json\s*','',raw); raw = re.sub(r'```\s*','',raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    sm = re.search(r'"max_particle_size_mm"\s*:\s*(\d+)', raw)
    gm = re.search(r'"grading"\s*:\s*"(\w+)"', raw)
    if sm and gm: return {'max_particle_size_mm': int(sm.group(1)), 'grading': gm.group(1)}
    return None

def run_eval(model, processor, manifest):
    model.eval()
    results=[]; cs=0; cg=0; vj=0; tt=0
    for i, entry in enumerate(manifest):
        img_path = os.path.join(TEST_DIR, entry['image'])
        if not os.path.exists(img_path): continue
        image = Image.open(img_path).convert('RGB')
        scale = min(MAX_DIM/max(image.size), 1.0)
        if scale < 1.0: image = image.resize((int(image.width*scale),int(image.height*scale)), Image.Resampling.LANCZOS)
        gsd = ORIGINAL_GSD * scale
        msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':EVAL_PROMPT.format(gsd=gsd)}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
        t=time.time()
        with torch.no_grad(): ids = model.generate(**inputs, max_new_tokens=128, temperature=0.7, do_sample=True)
        elapsed=time.time()-t; tt+=elapsed
        out = processor.batch_decode(ids[:,inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
        del inputs, ids; image.close(); torch.cuda.empty_cache()
        parsed = parse_response(out)
        gs=entry['max_particle_size_mm']; gg=entry['grading']; so=False; go=False
        if parsed:
            vj+=1; ps=parsed.get('max_particle_size_mm')
            if isinstance(ps,str): ps=int(ps) if ps.isdigit() else None
            if ps==gs: so=True; cs+=1
            if parsed.get('grading','').lower()==gg: go=True; cg+=1
        results.append({'image':entry['image'],'class':entry['class'],'gt_size':gs,'gt_grading':gg,
            'predicted':parsed,'raw':out,'size_correct':so,'grading_correct':go,'valid_json':parsed is not None,'time_s':round(elapsed,2)})
        if (i+1)%20==0:
            n=i+1; print(f'  [{n}/{len(manifest)}] Size:{cs}/{n}({cs/n*100:.0f}%) Grade:{cg}/{n}({cg/n*100:.0f}%)')
    return results, cs, cg, vj, tt

def eval_adapter(base_model, processor, adapter_path, manifest, label):
    """Load adapter from disk, eval, unload, free memory."""
    print(f'\nLoading {adapter_path}...')
    model = PeftModel.from_pretrained(base_model, adapter_path)
    results, cs, cg, vj, tt = run_eval(model, processor, manifest)
    model.unload()
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f'Unloaded. GPU 0: {torch.cuda.memory_allocated(0)/1e9:.1f} GB')
    return results, cs, cg, vj, tt

with open(TEST_MANIFEST) as f: test_manifest = json.load(f)
print(f'Test set: {len(test_manifest)} images')


### Quick Eval (9 images — 1 per class)

In [ ]:
quick_classes = sorted(set(e['class'] for e in test_manifest))
quick_manifest = [next(e for e in test_manifest if e['class']==cls) for cls in quick_classes]

for label, adapter in [('A: Direct', 'lora_direct'), ('B: Augmented', 'lora_augmented')]:
    print(f'\n{"="*50}')
    print(f'{label} — Quick eval (9 images)')
    print(f'{"="*50}')
    r, cs, cg, vj, tt = eval_adapter(base_model, processor, adapter, quick_manifest, label)
    n = len(r)
    print(f'Size: {cs}/{n} ({cs/n*100:.0f}%) | Grading: {cg}/{n} ({cg/n*100:.0f}%)')
    for x in r:
        p = x['predicted']
        ps = p.get('max_particle_size_mm','?') if p else '?'
        pg = p.get('grading','?') if p else '?'
        sv = '✓' if x['size_correct'] else '✗'
        gv = '✓' if x['grading_correct'] else '✗'
        print(f'  {x["class"]:>3} GT:{x["gt_size"]}mm/{x["gt_grading"]} Pred:{ps}mm/{pg} Size{sv} Grade{gv}')


### Full Eval (108 images)

In [ ]:
print('Evaluating Approach A on 108 images...')
r_a, cs_a, cg_a, vj_a, tt_a = eval_adapter(base_model, processor, 'lora_direct', test_manifest, 'A')
print(f'Done. {len(r_a)} images in {tt_a:.0f}s')

print('\nEvaluating Approach B on 108 images...')
r_b, cs_b, cg_b, vj_b, tt_b = eval_adapter(base_model, processor, 'lora_augmented', test_manifest, 'B')
print(f'Done. {len(r_b)} images in {tt_b:.0f}s')


## Step 6: Compare All Results

In [ ]:
def make_summary(label, results, cs, cg, vj, tt):
    n=len(results); both=sum(1 for r in results if r['size_correct'] and r['grading_correct'])
    return {'label':label,'n':n,'json':round(vj/n*100,1),'size':round(cs/n*100,1),
            'grading':round(cg/n*100,1),'both':round(both/n*100,1),'time':round(tt/n,2)}

sum_a = make_summary('LoRA Direct (18)', r_a, cs_a, cg_a, vj_a, tt_a)
sum_b = make_summary('LoRA SEAL (~144)', r_b, cs_b, cg_b, vj_b, tt_b)

rows = []
for mode, path in [('Qwen base (ZS)','../../task3-benchmarking/granulometry/benchmark_results_zero-shot.json'),
                    ('Qwen base (FS)','../../task3-benchmarking/granulometry/benchmark_results_few-shot.json'),
                    ('GPT-4.1 (FS)','../../task3-benchmarking/granulometry/benchmark_results_gpt41-few-shot.json')]:
    if os.path.exists(path):
        with open(path) as f: d=json.load(f)
        rows.append({'label':mode,'json':d['json_validity_pct'],'size':d['size_accuracy_pct'],
            'grading':d['grading_accuracy_pct'],'both':d['both_correct_pct'],'time':d['avg_inference_time_s']})
rows.extend([sum_a, sum_b])

print(f'{"Method":<24} {"JSON":>6} {"Size":>7} {"Grade":>7} {"Both":>7} {"Time":>6}')
print('='*59)
for r in rows:
    print(f'{r["label"]:<24} {r["json"]:>5.0f}% {r["size"]:>6.1f}% {r["grading"]:>6.1f}% {r["both"]:>6.1f}% {r["time"]:>5.1f}s')


## Visualization

In [ ]:
labels = [r['label'] for r in rows]; x = np.arange(len(labels)); w = 0.25
fig, ax = plt.subplots(figsize=(14, 6))
b1 = ax.bar(x-w, [r['size'] for r in rows], w, label='Size', color='steelblue')
b2 = ax.bar(x, [r['grading'] for r in rows], w, label='Grading', color='coral')
b3 = ax.bar(x+w, [r['both'] for r in rows], w, label='Both', color='green')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('%'); ax.set_ylim(0,110); ax.set_title('Task 4: LoRA Results vs Baselines')
ax.legend(); ax.axhline(y=33.3, color='gray', linestyle='--', alpha=0.5)
for bars in [b1,b2,b3]:
    for bar in bars: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{bar.get_height():.0f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.show()


## Save Results

In [ ]:
for label, s, results in [('direct', sum_a, r_a), ('augmented', sum_b, r_b)]:
    fname = f'results_{label}.json'
    with open(fname, 'w') as f:
        json.dump({'model':MODEL_ID,'adapter':f'lora_{label}','phase':'fine_tuned',
            'total_images':s['n'],'json_validity_pct':s['json'],'size_accuracy_pct':s['size'],
            'grading_accuracy_pct':s['grading'],'both_correct_pct':s['both'],
            'avg_inference_time_s':s['time'],'results':results}, f, indent=2)
    print(f'Saved {fname}')
print(f'\nAdapters: lora_direct/ and lora_augmented/')
print(f'Winner goes to Task 5 for quantization + edge deployment.')
